# 02 · ETL e Integração SIH + CNES

**Objetivo:** Carregar os dados brutos do SIH e CNES, filtrar internações por IAM (CID I21),realizar limpeza e integrar as duas bases em uma base de modelagem unificada.

**Inputs:**
- `data/input/SIH/*.csv` — registros de AIH convertidos do SIH
- `data/input/CNES/*.csv` — tabelas do CNES (ST, LT, EQ, SR, HB)
- `data/external/dicionario_SIH.json` — mapeamento de colunas do SIH
- `data/external/dicionario_CNES_*.json` — mapeamentos de colunas do CNES

**Outputs gerados:**
- `data/interim/sih_iam.csv` — internações por IAM (base limpa SIH)
- `data/interim/cnes_hospitais.csv` — base mestre de hospitais (CNES consolidado)
- `data/processed/base_modelagem.csv` — base final SIH × CNES pronta para modelagem

## 0. Configuração do Ambiente

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import sys

In [2]:
# Adiciona a raiz do projeto ao sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [80]:
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Caminhos ───────────────────────────────────────────────────────────
RAW_SIH  = Path(ROOT, 'data', 'input', 'SIH')
RAW_CNES = Path(ROOT, 'data', 'input', 'CNES')
INTERIM  = Path(ROOT, 'data', 'interim')
PROCESSED = Path(ROOT, 'data', 'processed')
EXTERNAL = Path(ROOT, 'data', 'external')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados:")
print(f"  SIH  : {RAW_SIH}")
print(f"  CNES : {RAW_CNES}")

Caminhos configurados:
  SIH  : /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH
  CNES : /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


## 1. Carregamento e Padronização do SIH

### 1.1 Leitura dos arquivos

Concatenamos **todos** os arquivos CSV disponíveis em `data/input/SIH/`.
O dicionário `dicionario_SIH.json` é usado para renomear as colunas para nomes legíveis.

In [82]:
arquivos_sih = sorted(RAW_SIH.glob('*.csv'))
print(f"Arquivos SIH encontrados: {len(arquivos_sih)}")
for f in arquivos_sih:
    print(f"  {f.name}")

Arquivos SIH encontrados: 12
  rdsp2501.csv
  rdsp2502.csv
  rdsp2503.csv
  rdsp2504.csv
  rdsp2505.csv
  rdsp2506.csv
  rdsp2507.csv
  rdsp2508.csv
  rdsp2509.csv
  rdsp2510.csv
  rdsp2511.csv
  rdsp2512.csv


In [5]:
# Carrega e renomeia colunas usando o dicionário externo
path_dicionario = Path(ROOT, 'data', 'external', 'dicionario_SIH.json')
with open(path_dicionario, 'r', encoding='utf-8') as f:
    schema = json.load(f)

rename_dict = {col["old_name"]: col["new_name"] for col in schema}

# Concatena todos os meses disponíveis
frames = []
for arq in arquivos_sih:
    df_tmp = pd.read_csv(arq, dtype=str, low_memory=False)
    df_tmp = df_tmp.rename(columns=rename_dict)
    frames.append(df_tmp)

df_sih = pd.concat(frames, ignore_index=True)
print(f"SIH carregado: {df_sih.shape[0]:,} registros × {df_sih.shape[1]} colunas")
df_sih.head(3)

SIH carregado: 2,948,801 registros × 114 colunas


,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,tipo_diag_sec_1,tipo_diag_sec_2,tipo_diag_sec_3,tipo_diag_sec_4,tipo_diag_sec_5,tipo_diag_sec_6,tipo_diag_sec_7,tipo_diag_sec_8,tipo_diag_sec_9,FONTE_ORC
0,350000,2025,01,01,46374500028366,3525100117847,1,11704840,354100,19840716,...,1,0,0,0,0,0,0,0,0,NaN
1,350000,2025,01,02,46374500028366,3524130275908,1,11741802,352210,20070606,...,1,1,1,1,1,0,0,0,0,NaN
2,350000,2025,01,02,46374500028366,3524130278427,1,11730000,353110,20030120,...,1,1,1,0,0,0,0,0,0,NaN


In [15]:
# identificação dos tipos das colunas
df_sih.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2948801 entries, 0 to 2948800
Columns: 114 entries, municipio_gestor to FONTE_ORC
dtypes: str(114)
memory usage: 2.5 GB


Todas  as colunas foram identificadas como tipo string, o que devera ser ajustado em breve.

### 1.2 Filtro CID — Internações por IAM

Filtramos registros cujo diagnóstico principal **ou** secundário inicie com `I21`
(Infarto Agudo do Miocárdio — CID-10), que é o foco do estudo.

In [21]:
df_sih["diagnostico_principal"]  = df_sih["diagnostico_principal"].astype(str)
df_sih["diagnostico_secundario"] = df_sih["diagnostico_secundario"].astype(str)

df_iam = df_sih[
    df_sih["diagnostico_principal"].str.startswith("I21") |
    df_sih["diagnostico_secundario"].str.startswith("I21")
].copy()

print(f"Total de internações: {len(df_sih)}")
print(f"Internações por IAM : {len(df_iam)} ({len(df_iam)/len(df_sih)*100:.1f}%)")

Total de internações: 2948801
Internações por IAM : 49047 (1.7%)


### 1.2 Seleção de Colunas Relevantes

In [22]:
df_iam.shape

(49047, 114)

#### Tratamento de Valores Ausentes
Campos administrativos do DataSUS frequentemente chegam como strings vazias ou `"0000"`.
Substituímos apenas **colunas de texto** (object), preservando zeros em variáveis
numéricas legítimas (ex.: `indicador_obito=0` = alta; `uti_mes_total=0` = sem UTI).

In [23]:
# Aplica replace somente em colunas de texto para não corromper variáveis numéricas/binárias
str_cols = df_iam.select_dtypes(include='str').columns
df_iam[str_cols] = df_iam[str_cols].replace(["", "0000", "000"], pd.NA)

# Converte colunas numéricas para tipo correto
num_cols = ["idade", "dias_permanencia", "indicador_obito",
            "uti_mes_total", "codigo_idade"]
for col in num_cols:
    if col in df_iam.columns:
        df_iam[col] = pd.to_numeric(df_iam[col], errors='coerce')

        # IDADE INCOERentE

print("Tratamento de nulos concluído.")

Tratamento de nulos concluído.


Ainda temos 114 colunas e nem todas serão nesessárias então iremos aplicar uma limpeza.
#### Remoção de colunas inutilizáveis

In [35]:
pct_nulos = df_iam.isnull().mean() * 100
print(pct_nulos.sort_values(ascending=False))

cnpj_mantenedora           55.33
diagnostico_secundario_1   51.46
FONTE_ORC                  15.79
cnpj_hospital              13.40
mes_competencia             0.00
                            ... 
tipo_diag_sec_4             0.00
tipo_diag_sec_6             0.00
tipo_diag_sec_7             0.00
tipo_diag_sec_8             0.00
tipo_diag_sec_9             0.00
Length: 91, dtype: float64


In [42]:
# ── Remoção de colunas inutilizáveis ─────────────────────────────────────────
# Colunas 100% nulas (sem informação alguma)
limite = 70  # %
cols_remover = pct_nulos[pct_nulos > limite].index
df_iam = df_iam.drop(columns=cols_remover)
print(f'Removidas {len(cols_remover)} colunas com mais de {limite}% de nulos')


# Filtro de idade: manter apenas adultos (>= 18 anos)
#    IAM pediátrico é evento raro e biologicamente distinto — excluído do escopo
df_iam['idade'] = pd.to_numeric(df_iam['idade'], errors='coerce')
n_antes = len(df_iam)
df_iam = df_iam[df_iam['idade'] >= 18].copy()
print(f'Registros pediátricos removidos (idade < 18): {n_antes - len(df_iam)}')
print(f'Base SIH-IAM adultos: {len(df_iam)} registros × {df_iam.shape[1]} colunas')

Removidas 0 colunas com mais de 70% de nulos
Registros pediátricos removidos (idade < 18): 0
Base SIH-IAM adultos: 48977 registros × 91 colunas


In [37]:
# Resumo de completude
nulls = pd.DataFrame({
    "qtd_nulos":  df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values("perc_nulos", ascending=False)

nulls[nulls["qtd_nulos"] > 0]

,qtd_nulos,perc_nulos
cnpj_mantenedora,27098,55.33
diagnostico_secundario_1,25204,51.46
FONTE_ORC,7732,15.79
cnpj_hospital,6563,13.40


### 1.3 Salvar Base SIH-IAM Intermediária

In [41]:
path_sih_interim = INTERIM / "sih_iam.csv"
df_iam.to_csv(path_sih_interim, index=False)
print(f"SIH-IAM salvo em: {path_sih_interim}  ({len(df_iam):,} registros)")

SIH-IAM salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam.csv  (48,977 registros)


## 2. Carregamento e Padronização do CNES

Carregamos as cinco tabelas do CNES usando dicionários JSON específicos para cada prefixo.

In [48]:
arquivos_cnes = sorted(RAW_CNES.glob('*.csv'))
print(f"Arquivos CNES encontrados: {len(arquivos_cnes)}")
for f in arquivos_cnes:
    print(f"  {f.name}")

Arquivos CNES encontrados: 5
  eqsp2512.csv
  hbsp2512.csv
  ltsp2512.csv
  srsp2512.csv
  stsp2512.csv


In [49]:
def load_cnes_custom(tipo, arquivos_cnes):

    tipo = tipo.lower()

    # encontra arquivo correspondente
    arq = next(
        (f for f in arquivos_cnes if f.name.lower().startswith(tipo)),
        None
    )

    if arq is None:
        raise FileNotFoundError(f'Arquivo {tipo} não encontrado')

    # dicionário
    path_dict = Path(
        ROOT,
        'data',
        'external',
        f'dicionario_CNES_{tipo.upper()}.json'
    )

    with open(path_dict, 'r', encoding='utf-8') as f:
        schema = json.load(f)

    rename_dict = {
        col["old_name"]: col["new_name"]
        for col in schema
    }

    df = pd.read_csv(
        arq,
        dtype=str,
        low_memory=False
    )

    # renomeia colunas
    df = df.rename(columns=rename_dict)

    df['arquivo_origem'] = arq.name
    df['tipo_cnes'] = tipo.upper()

    return df

In [88]:
df_st = load_cnes_custom('st', arquivos_cnes)
df_lt = load_cnes_custom('lt', arquivos_cnes)
df_eq = load_cnes_custom('eq', arquivos_cnes)
df_sr = load_cnes_custom('sr', arquivos_cnes)
df_hb = load_cnes_custom('hb', arquivos_cnes)

print(f"\nRegistros carregados por tabela:")
print(f"  ST (Estabelecimentos) : {df_st.shape}")
print(f"  LT (Leitos)           : {df_lt.shape}")
print(f"  EQ (Equipamentos)     : {df_eq.shape}")
print(f"  SR (Serviços)         : {df_sr.shape}")
print(f"  HB (Habilitações)     : {df_hb.shape}")


Registros carregados por tabela:
  ST (Estabelecimentos) : (109849, 210)
  LT (Leitos)           : (8350, 30)
  EQ (Equipamentos)     : (242363, 30)
  SR (Serviços)         : (177050, 34)
  HB (Habilitações)     : (6755, 34)


In [92]:
df_st

,codigo_cnes,codigo_municipio,cep_estabelecimento,cpf_cnpj_estabelecimento,tipo_pessoa,nivel_dependencia,cnpj_mantenedora,codigo_retencao_mantenedora,codigo_regiao_saude,codigo_micro_regiao_saude,...,regulacao_seguro_terceiro,regulacao_plano_publico,regulacao_plano_privado,AP07CV07,possui_atendimento_prestado,DT_ATUAL,competencia,natureza_juridica,arquivo_origem,tipo_cnes
0,0047406,350010,17800037,35723744000119,3,1,00000000000000,NaN,0209,NaN,...,0,0,0,0,1,202409,202512,2062,stsp2512.csv,ST
1,0081655,350010,17800057,00381929000108,3,1,00000000000000,NaN,R209,NaN,...,0,0,0,0,1,202508,202512,2062,stsp2512.csv,ST
2,0109789,350010,17803116,36060657000191,3,1,00000000000000,NaN,0209,NaN,...,0,0,0,0,1,202409,202512,2135,stsp2512.csv,ST
3,0109827,350010,17800005,48346595000168,3,1,00000000000000,NaN,0209,NaN,...,0,0,0,0,1,202409,202512,2135,stsp2512.csv,ST
4,0183555,350010,17800043,36363624000110,3,1,00000000000000,NaN,0209,NaN,...,0,0,0,0,1,202410,202512,2135,stsp2512.csv,ST
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109844,6470246,355730,99999999,00000000000000,3,3,67168856000141,NaN,NaN,NaN,...,0,0,0,0,1,202406,202512,1244,stsp2512.csv,ST
109845,6753442,355730,13857310,00000000000000,3,3,67168856000141,NaN,NaN,NaN,...,0,0,0,0,1,202512,202512,1244,stsp2512.csv,ST
109846,6811507,355730,13858300,00000000000000,3,3,67168856000141,NaN,NaN,NaN,...,0,0,0,0,1,202512,202512,1244,stsp2512.csv,ST
109847,6836542,355730,13857104,00000000000000,3,3,67168856000141,NaN,NaN,NaN,...,0,0,0,0,1,202511,202512,1244,stsp2512.csv,ST


### 2.2 Traduzir códigos de cnes para melhor entendimento

In [90]:
path_dicionario_lt = Path(EXTERNAL,'X')
map_leitos = pd.read_csv(path_dicionario_lt)

map_dict = dict(
    zip(
        map_leitos['codigo'].astype(str),
        map_leitos['descricao']
    )
)
df_lt['especialidade_leito_desc'] = (
    df_lt['codigo_especialidade_leito']
    .astype(str)
    .map(map_dict)
)
df_lt

FileNotFoundError: [Errno 2] No such file or directory: '/home/carolina/Documents/TCC Documentos/TCC/data/external/X'

### 2.2 pivot Tabelas CNES 

#### Remoção de Duplicidades (ST)

In [68]:
# Garante uma linha por hospital — mantém o registro mais recente
n_antes = len(df_st)
df_st = df_st.drop_duplicates(subset=['codigo_cnes'], keep='last')
print(f"ST: {n_antes} → {len(df_st)} registros únicos (removidas {n_antes - len(df_st):,} duplicatas)")

ST: 109849 → 109849 registros únicos (removidas 0 duplicatas)


#### Agregação de Leitos (LT)

In [69]:
# Pivot de leitos por tipo
df_lt['quantidade_leitos_existentes'] = pd.to_numeric(
    df_lt['quantidade_leitos_existentes'],
    errors='coerce'
).fillna(0)

df_lt_pivot = df_lt.pivot_table(
    index='codigo_cnes',
    columns='tipo_leito',
    values='quantidade_leitos_existentes',
    aggfunc='sum',
    fill_value=0
)

df_lt_pivot.columns = [
    f'leitos_{col}'
    for col in df_lt_pivot.columns
]

df_lt_pivot = df_lt_pivot.reset_index()

#### Agregação de Equipamentos (EQ)

In [70]:
# Pivot de equipamentos
df_eq['quantidade_em_uso'] = pd.to_numeric(
    df_eq['quantidade_em_uso'],
    errors='coerce'
).fillna(0)

df_eq_pivot = df_eq.pivot_table(
    index='codigo_cnes',
    columns='codigo_equipamento',
    values='quantidade_em_uso',
    aggfunc='sum',
    fill_value=0
)

df_eq_pivot.columns = [
    f'equip_{col}'
    for col in df_eq_pivot.columns
]

df_eq_pivot = df_eq_pivot.reset_index()

#### Serviços Especializados (SR) e Habilitações (HB)

In [71]:
# Serviços especializados
df_sr['flag_servico'] = 1

df_sr_pivot = df_sr.pivot_table(
    index='codigo_cnes',
    columns='codigo_servico_especializado',
    values='flag_servico',
    aggfunc='max',
    fill_value=0
)

df_sr_pivot.columns = [
    f'servico_{col}'
    for col in df_sr_pivot.columns
]

df_sr_pivot = df_sr_pivot.reset_index()

In [72]:
# Habilitações
df_hb['flag_habilitacao'] = 1

df_hb_pivot = df_hb.pivot_table(
    index='codigo_cnes',
    columns='codigo_habilitacao',
    values='flag_habilitacao',
    aggfunc='max',
    fill_value=0
)

df_hb_pivot.columns = [
    f'habilitacao_{col}'
    for col in df_hb_pivot.columns
]

df_hb_pivot = df_hb_pivot.reset_index()

In [75]:
print(df_lt_pivot.shape)
print(df_eq_pivot.shape)
print(df_sr_pivot.shape)
print(df_hb_pivot.shape)

(1462, 8)
(50460, 99)
(41011, 64)
(2282, 190)


## 3 Salvar Base Mestre de Hospitais (Interim CNES)

In [78]:
path_cnes_interim = INTERIM / "cnes_hospitais.csv"
df_cnes_final.to_csv(path_cnes_interim, index=False)
print(f"CNES Interim salvo em: {path_cnes_interim}  ({len(df_cnes_final):,} hospitais)")

NameError: name 'df_cnes_final' is not defined

## 4. Fusão Global: SIH-IAM × CNES

Fazemos um *Left Join* das internações por IAM com a base mestre de hospitais.
A chave de junção é `codigo_cnes`, normalizada em ambos os lados para evitar
divergências por zeros à esquerda ou espaços.

In [ ]:
# Padroniza a chave de merge nos dois DataFrames
if 'cnes' in df_iam.columns:
    df_iam = df_iam.rename(columns={'cnes': 'codigo_cnes'})

df_iam['codigo_cnes']        = df_iam['codigo_cnes'].astype(str).str.strip().str.zfill(7)
df_cnes_final['codigo_cnes'] = df_cnes_final['codigo_cnes'].astype(str).str.strip().str.zfill(7)

# Left Join: mantém todos os pacientes IAM
df_base_modelagem = pd.merge(df_iam, df_cnes_final, on='codigo_cnes', how='left')

# Log de match rate
n_sem_cnes = df_base_modelagem[df_cnes_final.columns.difference(['codigo_cnes'])[0]].isna().sum()
print(f"Total de internações IAM          : {len(df_base_modelagem):,}")
print(f"Sem correspondência no CNES       : {n_sem_cnes:,} ({n_sem_cnes/len(df_base_modelagem)*100:.1f}%)")
print(f"Com dados hospitalares do CNES    : {len(df_base_modelagem) - n_sem_cnes:,}")
print(f"Número de features da base final  : {df_base_modelagem.shape[1]}")

In [ ]:
# Exporta base de modelagem final
path_base_final = PROCESSED / "base_modelagem.csv"
df_base_modelagem.to_csv(path_base_final, index=False)
print(f"Base de modelagem salva em: {path_base_final}")
df_base_modelagem.sample(3)

In [ ]:
df_base_modelagem

---
## Resumo do Pipeline

| Etapa | Input | Output | Registros |
|-------|-------|--------|-----------|
| 1. Filtro IAM (SIH) | `data/input/SIH/*.csv` | `data/interim/sih_iam.csv` | ver acima |
| 2. Consolidação CNES | `data/input/CNES/*.csv` | `data/interim/cnes_hospitais.csv` | ver acima |
| 3. Fusão global | sih_iam + cnes_hospitais | `data/processed/base_modelagem.csv` | ver acima |

> **Próximos passos:** notebook `03_analise_exploratotia_visualizacao.ipynb` — análise exploratória e visualizações.